In [0]:
spark.sql("use CATALOG `databricks-pyspark` ")

In [0]:
spark.sql("use SCHEMA `databricks-pyspark-schema` ")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS employee_bronze (
            emp_id long,
            name string,
            salary long)
            using DELTA 
            TBLPROPERTIES("delta.enableChangeDataFeed"="true")
            
          """)

In [0]:
spark.sql("""
          insert into employee_bronze values (1,"John",10000),(2,"Mary",20000),(3,"Mike",30000)
          """)

In [0]:
%sql
describe history employee_bronze;

In [0]:
%sql
insert into employee_bronze values(5,"rin",42000)

In [0]:
%sql
update employee_bronze set salary = 50000 where emp_id = 2

In [0]:
%sql
delete from employee_bronze where emp_id = 3;

In [0]:
%sql
select * from employee_bronze;

In [0]:
cdf_df = spark.read.format("delta") \
     .option("readChangeFeed", "true") \
     .option("startingVersion", "2") \
         .table("employee_bronze")

In [0]:
cdf_df.show(truncate=False)

In [0]:
final_silver_df = cdf_df.filter("""
                                _change_type in ("update_postimage","insert","delete")
                                """).orderBy("_commit_version","emp_id")

In [0]:
final_silver_df.show()

In [0]:
%sql
describe history employee_bronze;

In [0]:
spark.sql("""
SELECT *
FROM table_changes('employee_bronze', 2)
ORDER BY _commit_version, emp_id
""").show(truncate=False)

In [0]:
final_silver_df = spark.sql("""
SELECT *
FROM table_changes('employee_bronze', 2)
WHERE _change_type IN ('insert', 'update_postimage', 'delete')
ORDER BY _commit_version, emp_id
""")

final_silver_df.show(truncate=False)